In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Task 1: Within-Subject EEG Classification

This notebook provides a starter scaffold for Task 1.

In the student release, Task 1 follows a Kaggle-style setup:
- `train.npz` is labeled
- `test.npz` is unlabeled
- you must build your own local validation protocol from the training set

What you must do:
- implement preprocessing
- run the experiment for all 4 required frequency bands
- compare local validation performance across bands
- choose one final approach and export predictions for the official test set


## Important Notes

- Do not use the official test set to tune your model.
- Do not fit preprocessing statistics on the official test set.
- Use the training set to create your own train/validation split.
- Run this notebook from the current Task 1 directory (`part1/task1`) so relative data paths resolve correctly.
- The final CSV must use header `id,label`.

## What This Block Does: Setup

This block prepares the basic environment for the rest of the notebook.

You can think of it as the notebook's control panel: it loads the tools we need, points to the dataset, lists the four required frequency bands, and sets the main training options.

It also fixes the random seed so the experiments are more stable and easier to reproduce.


In [15]:
from pathlib import Path
import csv
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset

SEED = 42
TASK1_DATA_DIR = Path("/content/drive/My Drive/515513_07_HW3_code/part1/task1/data")
TRAIN_FILE = TASK1_DATA_DIR / "train.npz"
TEST_FILE = TASK1_DATA_DIR / "test.npz"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError(
        "Task 1 data files not found. Expected: data/train.npz and data/test.npz under part1/task1. "
        "Please run this notebook from part1/task1."
    )

BANDS = {
    "alpha_mu": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "broad": (4.0, 40.0),
    "high_gamma": (70.0, 125.0),
}

TRAIN_CONFIG = {
    "batch_size": 64,
    "epochs": 50,
    "learning_rate": 1e-3,
    "val_fraction": 0.2,
}

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
DEVICE

device(type='cpu')

## What This Block Does: Dataset And File Loading

These blocks load the released Task 1 data and organize it into a form the training pipeline can use.

At a high level, this is the step where the notebook separates the labeled training data from the unlabeled test data and gets everything ready for later training and prediction.

The printed shapes are just a quick check to make sure the files were read correctly.


In [16]:
class EEGDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray | None = None, ids: np.ndarray | None = None):
        self.x = x.astype(np.float32)
        self.y = None if y is None else y.astype(np.int64)
        self.ids = None if ids is None else ids.astype(np.int64)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        sample = torch.from_numpy(self.x[index])
        if sample.ndim == 2:
            sample = sample.unsqueeze(0)

        if self.y is not None:
            label = torch.tensor(self.y[index], dtype=torch.long)
            return sample, label

        sample_id = torch.tensor(self.ids[index], dtype=torch.long)
        return sample, sample_id

In [17]:
def load_task1_train(data_file: Path):
    arrays = np.load(data_file, allow_pickle=True)
    required = ["x", "y"]
    missing = [key for key in required if key not in arrays]
    if missing:
        raise KeyError(f"Missing keys in {data_file}: {missing}")
    return arrays["x"], arrays["y"]


def load_task1_test(data_file: Path):
    arrays = np.load(data_file, allow_pickle=True)
    required = ["x"]
    missing = [key for key in required if key not in arrays]
    if missing:
        raise KeyError(f"Missing keys in {data_file}: {missing}")
    x = arrays["x"]
    ids = arrays["id"] if "id" in arrays else np.arange(len(x), dtype=np.int64)
    return x, ids


x_train_raw, y_train = load_task1_train(TRAIN_FILE)
x_test_raw, test_ids = load_task1_test(TEST_FILE)
print("train:", x_train_raw.shape, y_train.shape)
print("test :", x_test_raw.shape, test_ids.shape)

train: (16, 45, 1125) (16,)
test : (16, 45, 1125) (16,)


## TODO: Preprocessing

Implement your preprocessing pipeline here.

Minimum expectation:
- use the selected frequency band
- avoid leaking validation or test information into training
- keep the output shape compatible with EEGNet


In [18]:
def preprocess_train_val_test(
    x_train: np.ndarray,
    x_val: np.ndarray,
    x_test: np.ndarray,
    band: tuple[float, float],
):
    """
    TODO(student): implement preprocessing for one frequency band.

    Suggested contract:
    - inputs have shape (N, C, T)
    - return arrays with shape (N, C, T) or (N, 1, C, T)
    - fit any statistics on x_train only
    - apply the same fitted transformation to x_val and x_test
    """

    from scipy.signal import butter, filtfilt

    def filter(data, lowcut, highcut, fs = 250, order = 4):
        nyquist = 0.5 * fs
        low = lowcut / nyquist
        high = min(highcut / nyquist, 0.999)

        b, a = butter(order, [low, high], btype = 'band')

        filtered = np.zeros_like(data)

        for i in range(data.shape[0]):      # samples
            for j in range(data.shape[1]):  # channels
                filtered[i, j] = filtfilt(b, a, data[i, j])

        return filtered

    def normalize(train, val, test):
        m = train.mean()
        s = train.std() + 1e-8

        train = (train - m) / s
        val = (val - m) / s
        test = (test - m) / s

        return train, val, test

    low, high = band

    x_train = filter(x_train, low, high)
    x_val = filter(x_val, low, high)
    x_test = filter(x_test, low, high)

    x_train, x_val, x_test = normalize(x_train, x_val, x_test)

    x_train = np.expand_dims(x_train, axis = 1)
    x_val = np.expand_dims(x_val, axis = 1)
    x_test = np.expand_dims(x_test, axis = 1)

    return x_train, x_val, x_test

## What This Block Does: Model Definition

This block defines `EEGNet`, a compact neural network architecture designed for EEG signal classification.

EEGNet is commonly used in brain-computer interface and EEG decoding tasks because it is much smaller than many general-purpose deep networks, while still working well on time-series brain signals.

In this notebook, EEGNet is the classifier that takes the preprocessed EEG trial as input and predicts one of the four motor-imagery classes.


In [19]:
class EEGNet(nn.Module):
    def __init__(self, n_channels: int, n_samples: int, n_classes: int = 4, dropout: float = 0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8),
            nn.Conv2d(8, 16, kernel_size=(n_channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(self._infer_feature_dim(n_channels, n_samples), n_classes)

    def _infer_feature_dim(self, n_channels: int, n_samples: int) -> int:
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            features = self.features(dummy)
        return int(features.numel())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

## What This Block Does: Local Validation Split

This block splits the released training data into a training set and a validation set.

The validation split is used for local evaluation, so different frequency-band experiments can be compared on held-out data instead of only on the data used for training.

Using the same split across all experiments also makes the comparison more fair.


In [20]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=4, shuffle = True, random_state = SEED)
folds_indices = list(skf.split(x_train_raw, y_train))

print(f"local train: ({len(folds_indices[0][0])}, {x_train_raw.shape[1]}, {x_train_raw.shape[2]})")
print(f"local val  : ({len(folds_indices[0][1])}, {x_train_raw.shape[1]}, {x_train_raw.shape[2]})")

local train: (12, 45, 1125)
local val  : (4, 45, 1125)


## What This Block Does: DataLoaders And Training Utilities

This block groups the main training steps into small helper functions.

This is a common coding practice in machine learning projects. By separating data loading, training, evaluation, and prediction into different functions, the overall workflow becomes easier to understand and easier to maintain.

It also makes later experiments easier to adjust without rewriting the whole pipeline.


In [21]:
def build_dataloaders(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    x_test: np.ndarray,
    test_ids: np.ndarray,
):
    train_dataset = EEGDataset(x_train, y_train)
    val_dataset = EEGDataset(x_val, y_val)
    test_dataset = EEGDataset(x_test, y=None, ids=test_ids)

    train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=False)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_y.size(0)
        total_correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total_examples += batch_y.size(0)
    return total_loss / total_examples, total_correct / total_examples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        total_loss += loss.item() * batch_y.size(0)
        total_correct += (logits.argmax(dim=1) == batch_y).sum().item()
        total_examples += batch_y.size(0)
    return total_loss / total_examples, total_correct / total_examples


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    rows = []
    for batch_x, batch_ids in loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        pred = logits.argmax(dim=1).cpu().numpy()
        ids = batch_ids.numpy()
        rows.extend((int(sample_id), int(label)) for sample_id, label in zip(ids, pred))
    return rows

## What This Block Does: One Complete Experiment

This function is the main driver for one complete experiment under a single frequency band.

You can read it as the notebook's full pipeline for one band: prepare the data, train the model, track validation performance, keep the best version of the model, and then use that version to predict the official test set.


In [22]:
def run_single_experiment(band_name: str, band: tuple[float, float]):
    x_train, x_val, x_test = preprocess_train_val_test(
        x_train_base,
        x_val_base,
        x_test_raw,
        band,
    )
    train_loader, val_loader, test_loader = build_dataloaders(
        x_train,
        y_train_base,
        x_val,
        y_val_base,
        x_test,
        test_ids,
    )

    sample_shape = train_loader.dataset[0][0].shape
    _, n_channels, n_samples = sample_shape
    model = EEGNet(n_channels=n_channels, n_samples=n_samples).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CONFIG["learning_rate"])
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_val_acc = -1.0
    history = []
    for epoch in range(1, TRAIN_CONFIG["epochs"] + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    submission_rows = predict(model, test_loader, DEVICE)
    result = {
        "band": band_name,
        "low_hz": band[0],
        "high_hz": band[1],
        "best_val_acc": best_val_acc,
    }
    return result, pd.DataFrame(history), submission_rows

## Compare The Four Required Bands

This block runs the same pipeline on each of the four required frequency bands.

The goal is to see how performance changes when only the frequency band is changed.

This gives a direct comparison across the four band settings.


In [23]:
results = []
histories = {}
submissions = {}

for band_name, band in BANDS.items():
    print(f"Running band: {band_name} ({band[0]}-{band[1]} Hz)")

    fold_accs = []

    for fold_idx, (train_idx, val_idx) in enumerate(folds_indices):
        x_train_base, y_train_base = x_train_raw[train_idx], y_train[train_idx]
        x_val_base, y_val_base = x_train_raw[val_idx], y_train[val_idx]

        result, history_df, submission_rows = run_single_experiment(band_name, band)
        fold_accs.append(result["best_val_acc"])

    avg_val_acc = np.mean(fold_accs)

    results.append({
        "band": band_name,
        "low_hz": band[0],
        "high_hz": band[1],
        "best_val_acc": avg_val_acc
    })

    histories[band_name] = history_df
    submissions[band_name] = submission_rows

results_df = pd.DataFrame(results).sort_values("best_val_acc", ascending=False)
results_df

Running band: alpha_mu (8.0-13.0 Hz)
Running band: beta (13.0-30.0 Hz)
Running band: broad (4.0-40.0 Hz)
Running band: high_gamma (70.0-125.0 Hz)


,band,low_hz,high_hz,best_val_acc
2,broad,4.0,40.0,0.6250
0,alpha_mu,8.0,13.0,0.5625
3,high_gamma,70.0,125.0,0.4375
1,beta,13.0,30.0,0.3750


## Export Submission

Choose one band or one final pipeline design and export its predictions for the official test set.

In [24]:
def write_submission(rows, output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        writer.writerows(rows)


best_band = results_df.iloc[0]["band"]
output_path = Path(f"task1_{best_band}_submission.csv")
write_submission(submissions[best_band], output_path)
print(f"Wrote submission to {output_path}")

Wrote submission to task1_broad_submission.csv


## Report Checklist

- compare local validation performance across the 4 bands
- state the optimizer, learning rate, epochs, batch size, and validation protocol you used
- explain which band worked best in your local experiments and why
- state which final pipeline you used to generate the official submission file